# OmniTranscriber Minimal MVP - Colab GPU

**What it does**
- Transcribes long audio/video with Whisper on Google Colab GPU
- Keeps the core MVP only: batch input, timestamped transcript, subtitles, JSON, and ZIP download
- Outputs: `.txt` timestamped transcript, `.srt` & `.vtt` subtitles, `.json` segments

**How to run**
1. In Colab: `Runtime -> Change runtime type -> GPU` (T4/A100 is great)
2. Run each cell from top to bottom
3. Upload your MP3/MP4 or mount Google Drive and transcribe


In [ ]:
# 🔎 Check GPU (optional)
!nvidia-smi || echo "No GPU visible. You can still run on CPU, just slower."


In [ ]:
# 📦 Install dependencies (Whisper, PyTorch, FFmpeg)
%%bash
pip -q install -U openai-whisper ffmpeg-python >/dev/null
pip -q install -U torch torchvision torchaudio >/dev/null
apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null
echo '✅ Installed Whisper, Torch, and FFmpeg'


In [ ]:
# 🔗 (Optional) Mount Google Drive instead of manual upload
from google.colab import drive
drive.mount('/content/drive')
# Then set input_path to a Drive path, e.g., '/content/drive/MyDrive/Colab/Input'

In [ ]:
# 📂 Define Input and Output Paths
# Make sure these directories exist in your Google Drive.
input_path = '/content/drive/MyDrive/Colab/Input'
output_path = '/content/drive/MyDrive/Colab/Output'

# IMPORTANT: If you encounter an 'OutOfMemoryError' during transcription,
# your chosen model ('large-v3') might be too big for your GPU's memory.
# Consider changing 'model_size' in the next cell to a smaller model like 'medium.en' or 'small.en'.
# For example: model_size = 'medium.en'

In [ ]:
# 📂 Batch transcribe *all* uploaded files
import whisper, json, os, glob

def srt_ts(t):
    h = int(t // 3600); m = int((t % 3600)//60); s = int(t % 60); ms = int((t-int(t))*1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
def vtt_ts(t):
    h = int(t // 3600); m = int((t % 3600)//60); s = int(t % 60); ms = int((t-int(t))*1000)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

model_size = 'large-v3'
language   = None
domain_prompt = None # Initialize domain_prompt to prevent NameError
initial_prompt = domain_prompt
model = whisper.load_model(model_size)

os.makedirs(output_path, exist_ok=True)

# Get all files from the input_path
# You might need to adjust the pattern (e.g., '*.mp3', '*.mp4') based on your file types
input_files = glob.glob(os.path.join(input_path, '*.*'))

if not input_files:
    print(f"No files found in {input_path}. Please ensure files are in the directory and Google Drive is mounted correctly.")
else:
    for fname in input_files:
        print('\nTranscribing:', fname)
        result = model.transcribe(fname, language=language, initial_prompt=initial_prompt, verbose=True)
        stem = os.path.splitext(os.path.basename(fname))[0]
        txt_path  = os.path.join(output_path, f'{stem}.txt')
        srt_path  = os.path.join(output_path, f'{stem}.srt')
        vtt_path  = os.path.join(output_path, f'{stem}.vtt')
        json_path = os.path.join(output_path, f'{stem}.json')

        timestamped_lines = []
        for seg in result.get('segments', []):
            text = seg.get('text', '').strip()
            if text:
                timestamped_lines.append(f"[{vtt_ts(seg['start'])} --> {vtt_ts(seg['end'])}] {text}")
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(timestamped_lines).strip() or result.get('text','').strip())
        srt_lines = []
        for i, seg in enumerate(result.get('segments', []), start=1):
            srt_lines += [str(i), f"{srt_ts(seg['start'])} --> {srt_ts(seg['end'])}", seg['text'].strip(), ""]
        with open(srt_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(srt_lines))
        vtt = ['WEBVTT','']
        for seg in result.get('segments', []):
            vtt += [f"{vtt_ts(seg['start'])} --> {vtt_ts(seg['end'])}", seg['text'].strip(), ""]
        with open(vtt_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(vtt))
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print('Saved to:', txt_path, srt_path, vtt_path, json_path)
    print(f'\n✅ All done! Files are in {output_path}')

In [ ]:
%%bash
pip -q install -U openai-whisper >/dev/null
echo '✅ Re-installed openai-whisper'

In [ ]:
# ⬇️ Download outputs as a zip
import shutil
from google.colab import files

# Create a zip archive of the output directory
archive_name = 'transcripts'
shutil.make_archive(archive_name, 'zip', output_path)

# Download the zip file
files.download(f'{archive_name}.zip')